# AI 트렌드 사전과제

## 1. AI 에이전트
### 정의
- 정의 : 사용자가 정한 목표 달성을 위해 AI 모델을 활용해서 *환경과 상호작용하는 시스템*, **추론+계획+도구를 사용한 실행이 하나로 맞물린다**
### 에이전티 AI의 작동 방식 (ReAct)
- 생성형 ai는 프롬프트에 맞는 결과만 출력하고 끝나지만 에이전틱 ai는 목표를 달성할 때 까지 '추론과 행동을 반복한다.'
- 핵심작동방식 : reasoning + acting (생각-> 행동->관찰->생각)
    - 해당 리엑트 단계에서도 모델은 도구를 직접 실행하지 않는다. (실행은 앱에서)

### 다중 에이전트 
- MSA의 구조 : 여러 에이전트가 역할을 나눠서 함께 일하는 구조
    - 오케스트레이터 (요청을 작은 일로 쪼개서 역할을 분담) + 서브 에이전트 (맡은 역할만 해냄)
    - 역할을 나누는 이유 : 스스로 실수를 잡기 어려움 + 동시에 여러 갈래를 조사
    - 역할을 나누는 경우 : 고정 워크플로우 (순서가 정해진 단순 작업), 단일 에이전트 (도구 몇 개로 해결되는 작업), 다중 에이전트 (따로 조사해서 합칠 수 있는 작업)
    - 비용과 성능의 조합 = 역할마다 필요한 능력에 맞게 급이 다른 모델을 섞어 씀
        - 오케스트레이터, 검색/정리봇. 코드 검증봇

## 2. MCP (model context protocol)
### MCP 이전 -> 표준이 왜 필요한가?
- function calling/ tool calling : 모델이 구조화된 요청을 내놓는 기능 (실제 실행과 결과 전달은 앱이) 
    - 앱 -> 모델 : 질문과 사용 가능한 함수 목록
    - 모델 : 어떤 함수를 어떤 값으로 부를지 적어냄
    - 앱 : 그 함수를 실제 실행
    - 앱 -> 모델 : 실행 결과 전달해서 모델이 답 작성
- n x m 작업량 문제
    - 표준이 없을 때 : 도구 하나 늘어날 때 작업량도 늘어남
    - 표준이 생기면 : 앱은 MCP 클라이언트 1번, 도구는 MCP 서버 1번만 -> n+m개 (모두 같은 규격을 사용해서 어떤 앱이든 바로 연결돼서 필요한 작업이 줄어듦)
- 오해 바로 잡기 : function calling과 MCP는 경쟁 관계가 아니다. (실제 시스템은 둘 다 사용하는 것이다.: MCP로 도구를 가져오고 -> 모델이 function calling으로 하나를 고른 뒤 -> 앱이 다시 MCP로 실행)

### MCP의 등장 
- MCP : LLM 앱과 외부 데이터/도구가 통신하는 개방형 표준 프로토콜이다.
    - nxm -> n+m : 서버를 한 번 만들면 MCP를 지원하는 모든 앱에서 동작
    - 발견의 표준화 : 쓸 수 있는 도구 물어보면 서버가 목록을 반환함
    - 원격 서버 인증 통일 : 인증이 필요하면 로그인 표준을 따름

### MCP의 구조
- 역할
    - 1. host: 사용자가 쓰는 앱
    - 2. client : 호스트 내 통신 담당
    - 3. server : 도구와 데이터를 MCP 규격으로 내놓는 프로그램
- 기본 요소 (서버가 내놓을 수 있는 기능 종류)
    - tools : 실행 가능한 행동
    - resources : 읽기 전용 데이터
    - prompts : 재사용 가능한 탬플릿 <br>
    : RAG가 참고할 문서를 Resource로 넘길 수 있음

### 작동 흐름
- AI가 외부 도구를 쓸 때
    - 질문 전 -> 미리 준비 (연결(로그인), 도구 목록 요청 및 응답)
    - 질문 후 -> 사용자가 호스트에게 질문하고 호스트가 모델에게 전달한다. 이후 도구를 선택하고 호출해서 실제 실행한다. 결과를 전달하고 답변을 하며 과정이 마무리 된다.
    - 모델이 하는 일 : 도구를 고르고 답을 쓴다
    - 연결 및 호출 실행 : 호스트와 클라이언트, 서버 담당이다.

### MCP 의 한계와 주의점
- 보안은 호스트가 책임진다. 사용자 동의, 서버에 넘길 권한 범위 그리고 허용할 도구를 선택하는 것까지 모두 호스트의 몫이다.
- 연결이 쉬워진 만큼 공격받을 틈이 넓어진다...
    - 신뢰할 수 없는 서버를 붙이는 위험
    - 도구 설명을 통한 주입 (도구 오염 : 도구 설명 내에 모델에게만 보이는 지시 숨기는 공격)
    - 너무 넓은 권한 연동의 위험 -> 권한이 넓을수록 프롬프트 주입 피해가 커진다. (도구가 가져온 내용 안에 지시가 섞여서 들어오는 것이다.)
- 안전하게 외부 도구 사용하는 방법
    - 공식, 검증된 서버만 연결
    - 필요한 권한만 주기
    - 되돌리기 어려운 호출은 사람이 승인
    - 처음 연결하는 서버는 도구 설명 직접 확인

# 3. 데이터 구조화
### 왜 구조화가 필요할까 
- vanilla RAG의 한계 : 청크들 사이의 관계를 바닐라 RAG가 표현하지 못하는 것이다.
    - TOP-K : 질문과의 유사도가 높은 k개의 문서 조각만 선택해서 LLM에 전달하는 방식
        - 바닐라 래그는 조각을 하나씩 따로 채점
        - 그런데 정답에 필요한 조각이 질문과 비슷하지 않는 경우가 대부분이다.
        - 그래서 top k 딜레마가 생기게 된다. 

### 그래프 구조와 팔란티어 온톨로지
- 그래프 자료 구조 : 데이터를 노드와 엣지로 표현한다.
- 팔란티어 온톨로지 : 그래프 구조를 기업 **전체**에 적용해서 흩어진 사내 데이터를 하나의 의미 있는 관계망으로 묶은 것
- 구성요소
    - 객체 타입 : 테이블의 스키마
    - 객체 : 테이블의 한 행 -> 여러 객체의 모음 = 객체 세트
    - 속성 : 테이블의 열과 같다 (객체의 특징)
    - 속성값 : 같이 특정 객체가 갖는 속성의 실제 값
    - 공유 속성 : 여러 객체 타입에서 공통으로 사용되는 속성
    - 링크 타입 : 관계의 스키마
    - 링크 : 실제 관계의 인스턴스
    - *액션타입* (일반 DB와 가장 큰 차이점) : 실제 변경과 행동까지 온톨로지 위에서 정의
    - 역할 : 직급 부서 지역에 따라 접근 권한을 객체 단위까지 세밀하게 제어할 수 있음 , 사람처럼 에이전트도 권한을 부여받아서 그 범위 내에서만 데이터 조회 및 실행이 가능

### 실전에서는?
- 사례 : Neo4j, 사내 챗봇과 에이전트 구축
- light RAC vs GRAPH RAG : graph는 커뮤니티 요약을 미리 만들어두지만 light는 이 단계를 생략함
- graphify : 폴더를 읽어서 '무엇이 무엇과 어떻게 연결되는가'를 그래프로 만들어주는 오픈 소스 도구
    - claude obsidian은 클로드 코드를 사용해서 지식을 축적합니다.
    - claude obsidian : 클로드 코드 + obsidian 기반의 위키형 지식 그래프
- 온톨로지 (팔란티어)
    - 의료기기 제조사, 수십개 공급업체와의 납품 일정, 여러 공장의 생산 라인 상태, 전 세계 창고 입출고 현황, 고객사별 납품 계약과 수명 주기 등 수백개의 데이터 소스를 하나의 온톨로지로 통합
    - -> 각각의 시스템을 따로 찾아보지 않고 관계를 통해 추론이 가능함.

# 4. XAI
### 기존 AI가 답하지 못하는 문제
- "왜 그런 결정을 내렸나요?" => 그래서 등장한 것이 **설명 가능한 인공지능 XAI**
### 블랙박스가 왜 문제일까?
- 입력을 넣으면 결과가 나오지만 과정을 볼 수 없는 '블랙박스' -> 문제가 되는 이유는 **규제 때문**
    - 의료 (진단의 근거), 금융 (선택에 대한 이유 제공), 자율주행 (사고 시 책임소재)
- + 실용적 이유 (어디가 문제인지 디버깅하기 위해 내부를 봐야함)
### 기존 해결 시도들
- 회귀분석 : 완전한 화이트 박스
    - 그러나 복잡한 세상에선 충분한 정확도 내기 어려움
- SHAP /LIME : 사후적 설명 - 어떤 입력 특성이 결과에 얼마나 기여?
    - 그러나 진짜 이유가 아닐 수 있다. 역으로 추정하는 방법이기 때문...
- LLM + RAG : 블랙박스로 블랙박스 설명 (왜 이런 결정을 내렸을까요? 하고 묻는 방식)
    - 그럴듯한 설명일 수 있으나 실제 내부 연산과 일치한다는 보장이 없다.
### 기계적 해석 가능성
- 블랙박스 자체를 열어버리자! -> 기계적 해석 가능성 (모델이 어떻게 그 답에 도달하는가)
- 일반 XAI와의 차이 : 내부를 직접 해부하는 구조적 성격을 가졌다. (가중치 회로 수준)
- 핵심 기법 : 활성값 패칭 - *모델 내부의 어느 위치가 답을 결정하는가?*
    - 입력 2개 -> B 처리하면서 A의 값으로 바꿔 끼운다. -> 출력의 변화를 본다

### 핵심 개념 3가지
- 모델 안에는 무엇이 있는가 (특징) : 모델이 입력에서 알아챈 개념
    - 개별 뉴런이나 방향 벡터에 저장된다. 입력에 해당 개념이 등장하면 활성값이 커진다. 
    - 그러나 서로 상관없는 개념이 섞인 경우가 많다.
- 그것들이 어떻게 연결? (회로) : 여러 뉴런이 연결돼서 특정 기능을 수행하는 서브네트워크
- 왜 해석이 어려울까? (중첩) : 하나의 뉴런이 여러 개념을 중첩해서 저장한다. -> 뉴런 하나가 개념 하나가 아니기 때문에 뉴런을 봐도 무슨 개념인지 한번에 알 수는 없다. => 이를 해결하기 위해 나온 도구가 SAE이다.

### 실제 연구사례
- 수십억개의 파라미터를 가진 거대 모델의 핵심 능력이 2개의 어텐션 헤트로만 구현된다
- 중첩 문제를 해결 위해 SAE 도입 -> 특정 feature이 모델 자기 인식까지 영향을 줄 수 있다는 것을 보여준 실험이 있다.

### 현재 중요한 이유?
- AI 안전과의 연결 : 모델이 커지고 의사결정을 많이 위임할수록 내부를 이해하는 것이 중요해진다.
- 투자 현황 : 기계적 해석 가능성 분야에 주요 기관들이 집중 투자를 하고 있다. (엔트로픽, 딥마인드, redwood research)